# Kirishima elastic 2D benchmark: FD3, OPT3/4/5 and SPECFEM2D

This notebook consumes the case-specific solver-neutral bundles exported by `SimuKirishima.ipynb`. Generate `:none`, `:flat`, and `:topography` in that order. The first case has absorbing boundaries on every side; the latter two apply flat and Kirishima traction-free surfaces. SPECFEM remains an external executable and is not a flexOPT dependency.

In [ ]:
import Pkg
flexopt_root = let
    marker(dir) = isfile(joinpath(dir, "Project.toml")) &&
                  isfile(joinpath(dir, "src", "commonBatchs.jl"))
    candidates = String[]
    haskey(ENV, "FLEXOPT_ROOT") && push!(candidates, ENV["FLEXOPT_ROOT"])
    directory = abspath(pwd())
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    index = findfirst(marker, candidates)
    isnothing(index) && error("Cannot locate flexOPT; set ENV[\"FLEXOPT_ROOT\"]")
    abspath(candidates[index])
end
Pkg.activate(flexopt_root)
using JLD2, CairoMakie, Statistics
include(joinpath(flexopt_root, "src", "specfemBenchmark.jl"))
using .specfemBenchmark
@show Base.active_project() Threads.nthreads() specfem2d_status()


## Load the common numerical experiment

The bundle fixes the physical model, source, receivers and FD/OPT traces. This prevents accidental comparison of different grids, source frequencies or surface definitions.

In [ ]:
availableSurfaceCases = (:none, :flat, :topography)
selectedSurfaceCase = :flat
selectedSurfaceCase in availableSurfaceCases || error(
    "selectedSurfaceCase must be :none, :flat or :topography")
bundlePath = joinpath(flexopt_root, "data",
    "KirishimaElastic2D_$(selectedSurfaceCase).jld2")
isfile(bundlePath) || error("Generate the $(selectedSurfaceCase) bundle in SimuKirishima.ipynb first")
benchmark = load(bundlePath)
@show benchmark["free_surface_case"] size(benchmark["vp"])
@assert Symbol(benchmark["free_surface_case"]) === selectedSurfaceCase
@assert size(benchmark["vp"]) == size(benchmark["vs"]) == size(benchmark["rho"])
@assert length(benchmark["surface_z"]) == length(benchmark["x"])


In [ ]:
# SPECFEM's tomography file is rectangular although the spectral mesh ends at
# the free surface. Fill the unused samples above a topographic interface with
# the shallowest valid solid value; they are never sampled by the mesh.
vpSPECFEM = copy(benchmark["vp"])
vsSPECFEM = copy(benchmark["vs"])
rhoSPECFEM = copy(benchmark["rho"])
for ix in axes(vpSPECFEM, 1)
    valid = findall((@view vsSPECFEM[ix, :]) .> 0)
    isempty(valid) && error("column $ix contains no elastic material")
    top = last(valid)
    for field in (vpSPECFEM, vsSPECFEM, rhoSPECFEM)
        field[ix, top+1:end] .= field[ix, top]
    end
end
@assert minimum(vpSPECFEM) > 0
@assert minimum(vsSPECFEM) > 0
@assert minimum(rhoSPECFEM) > 0


## Generate and run the independent SPECFEM2D case

The generated case lives under `flexOPT/data/specfem2d_benchmarks`; the external SPECFEM checkout is never modified.

In [ ]:
caseName = "Kirishima_$(selectedSurfaceCase)"
caseDirectory = joinpath(flexopt_root, "data", "specfem2d_benchmarks", caseName)
dxBenchmark = minimum(diff(benchmark["x"]))
dzBenchmark = minimum(diff(benchmark["z"]))
dtSPECFEM = 0.15 * min(dxBenchmark, dzBenchmark) / maximum(vpSPECFEM)
specfemCase = prepare_specfem2d_case(
    caseDirectory,
    benchmark["x"], benchmark["z"],
    vpSPECFEM, vsSPECFEM, rhoSPECFEM,
    benchmark["surface_z"];
    source=benchmark["source"],
    receivers=benchmark["receiver_x"],
    duration=benchmark["duration"],
    dt=dtSPECFEM,
    f0=benchmark["source_frequency"],
    source_factor=benchmark["source_force_amplitude"],
    source_angle=0.0,  # vertical force in SPECFEM2D
    receiver_z=mean(benchmark["receiver_z"]),
    free_surface=selectedSurfaceCase !== :none,
)
@show specfemCase.case_directory dtSPECFEM


In [ ]:
# Set false when inspecting an already completed case.
runSPECFEM2D = true
specfemRun = runSPECFEM2D ?
    run_specfem2d_case(specfemCase.case_directory) :
    (output=specfemCase.output,)
@show specfemRun.output


## Compare surface velocity traces

Each trace is interpolated onto a common time interval for the metrics. Correlation diagnoses phase/polarity; the relative error is computed after one optimal scalar amplitude correction, so it does not hide waveform differences.

In [ ]:
# Prefixes can be CXZ, FXZ, etc. depending on SPECFEM's channel convention;
# select the physical vertical component and order it by station id.
specfemVerticalFiles = find_specfem2d_traces(
    specfemRun.output; component=:z)
@assert length(specfemVerticalFiles) == length(benchmark["receiver_x"])
receiver = cld(length(specfemVerticalFiles), 2)
fdTrace = (time=benchmark["fd_time"],
           values=benchmark["fd_traces"][:, receiver])
optTrace = (time=benchmark["opt_time"],
            values=benchmark["opt_traces"][:, receiver])
specfemTrace = read_specfem2d_trace(specfemVerticalFiles[receiver])
metricsOPT = waveform_metrics(fdTrace, optTrace)
metricsSPECFEM = waveform_metrics(fdTrace, specfemTrace)
@show metricsOPT.correlation metricsOPT.relative_error
@show metricsSPECFEM.correlation metricsSPECFEM.relative_error
peakVelocity = (
    FD3=maximum(abs, fdTrace.values),
    OPT3=maximum(abs, optTrace.values),
    SPECFEM2D=maximum(abs, specfemTrace.values),
)
peakRatioToFD3 = map(value -> value / peakVelocity.FD3, peakVelocity)
@show peakVelocity peakRatioToFD3
benchmarkPlot = plot_solver_benchmark(
    (FD3=fdTrace, OPT3=optTrace, SPECFEM2D=specfemTrace);
    title="Kirishima $(benchmark["free_surface_case"]) — receiver $(receiver)",
)
benchmarkPlot.figure

# Raw vertical-velocity amplitudes. This comparison is meaningful only
# because all three solvers now use the same 1e10 N/m vertical line force.
rawAmplitudePlot = plot_solver_benchmark(
    (FD3=fdTrace, OPT3=optTrace, SPECFEM2D=specfemTrace);
    normalize=false,
    title="Raw amplitude — same vertical line force",
)
rawAmplitudePlot.figure


## Interpretation order

1. Start with `:flat`: disagreement here indicates source normalization, temporal convention or numerical dispersion—not topography.
2. Compare arrival times before amplitudes.
3. Repeat with `:topography` only after the flat case is consistent.
4. SPECFEM is a high-order reference, not an automatic ground truth: verify its mesh resolution and reported minimum period in `OUTPUT_FILES`.